# WDS Phase 1 — Auto-Label Dataset (Grounding DINO)

**What this does:** runs Grounding DINO (zero-shot detector) over your workstation images on Drive, writes YOLO-format labels, splits into train/valid, and emits a `data.yaml` ready for YOLOv8 training in Phase 2.

**Runtime:** ~45–90 min on free Colab T4 GPU. Cost: $0.

**Output (in your Drive):**
```
wds_dataset/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/
```

**Classes (9):** `hand, glove, person, chair, paint, tray, container, tool, workpiece` — matches your Roboflow project.

## Step 1 · Switch runtime to T4 GPU

Top menu: **Runtime → Change runtime type → T4 GPU → Save.**

Run the cell below to confirm. If the assert fails, you're on CPU — switch the runtime.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU → Save.'
print(f'GPU OK: {torch.cuda.get_device_name(0)}')

## Step 2 · Install dependencies

Heavy install (~5 min). Only run once per Colab session.

In [ ]:
!pip install -q ultralytics==8.3.30 supervision==0.25.0

## Step 3 · Mount Google Drive

Run this cell and approve access in the popup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 4 · Make your shared folder visible to Colab

Your image folder was shared via link, but Colab can only see folders that live in *your* Drive. Two options:

**Option A (recommended) — Add shortcut**
1. Open the share link in your browser: https://drive.google.com/drive/folders/1qB9oOl6XPfyiyEK2-SBlNzJtotLK5w6s
2. Right-click the folder name in the Drive header → **Add shortcut to Drive** → place at the top level ("My Drive").
3. The shortcut will appear at `/content/drive/MyDrive/<folder_name>`.

**Option B — Copy folder ID into the next cell** (if Option A doesn't work)

Then run the cell below to see what's in your MyDrive.

In [ ]:
import os
for name in sorted(os.listdir('/content/drive/MyDrive')):
    print(name)

## Step 5 · Configure paths and class prompts

**Edit `INPUT_DIR`** to the folder you just added (the name shown above). The `ONTOLOGY` dict maps the prompt Grounding DINO sees → the class name written to YOLO labels.

In [ ]:
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────
INPUT_DIR  = '/content/drive/MyDrive/wds_images'      # <-- folder with your 5500 .jpgs
OUTPUT_DIR = '/content/drive/MyDrive/wds_dataset'     # <-- labels will be written here

# ── DO NOT EDIT BELOW ────────────────────────────────────────
ONTOLOGY = {
    # prompt for Grounding DINO    →   class name in YOLO labels
    'human hand':                       'hand',
    'rubber work glove':                'glove',
    'person':                           'person',
    'office chair':                     'chair',
    'paint can':                        'paint',
    'plastic tray':                     'tray',
    'plastic container':                'container',
    'hand tool':                        'tool',
    'workpiece':                        'workpiece',
}
CLASS_NAMES = list(ONTOLOGY.values())
print('YOLO class IDs:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

assert Path(INPUT_DIR).exists(), f'INPUT_DIR not found: {INPUT_DIR}'
imgs = sorted(list(Path(INPUT_DIR).glob('*.jpg')) + list(Path(INPUT_DIR).glob('*.png')))
print(f'\nFound {len(imgs)} images in {INPUT_DIR}')
assert len(imgs) > 0, 'No images found. Check INPUT_DIR.'

## Step 6 · Run auto-labeling

This is the long step. ~45–90 min for 5500 images on T4. The first call also downloads Grounding DINO weights (~700 MB). Keep this Colab tab active — Colab kills idle sessions.

Output goes straight to your Drive, so a session crash mid-way doesn't lose work — re-run and it skips already-labeled images.

In [ ]:
from ultralytics import YOLOWorld
from pathlib import Path
import shutil, random, yaml
from tqdm.auto import tqdm

CONF_THRESH = 0.20

model = YOLOWorld('yolov8s-world.pt')
model.set_classes(CLASS_NAMES)

out = Path(OUTPUT_DIR)
for split in ('train', 'valid'):
    (out / split / 'images').mkdir(parents=True, exist_ok=True)
    (out / split / 'labels').mkdir(parents=True, exist_ok=True)

random.seed(42)
all_imgs = sorted(Path(INPUT_DIR).glob('*.jpg'))
random.shuffle(all_imgs)
split_idx = int(len(all_imgs) * 0.8)
splits = {'train': all_imgs[:split_idx], 'valid': all_imgs[split_idx:]}

def to_yolo(box, w, h):
    x1, y1, x2, y2 = box
    return (x1+x2)/2/w, (y1+y2)/2/h, (x2-x1)/w, (y2-y1)/h

skipped = []
for split_name, paths in splits.items():
    img_dir   = out / split_name / 'images'
    label_dir = out / split_name / 'labels'
    for img_path in tqdm(paths, desc=split_name):
        out_label = label_dir / f'{img_path.stem}.txt'
        out_image = img_dir   / img_path.name
        if out_label.exists() and out_image.exists():
            continue
        try:
            result = model.predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
        except Exception as e:
            skipped.append(img_path.name)
            continue
        h, w = result.orig_shape
        lines = []
        if result.boxes is not None:
            for b in result.boxes:
                cls = int(b.cls.item())
                cx, cy, bw, bh = to_yolo(b.xyxy[0].tolist(), w, h)
                lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        out_label.write_text('\n'.join(lines))
        if not out_image.exists():
            shutil.copy(img_path, out_image)

(out / 'data.yaml').write_text(yaml.dump({
    'path':  str(out),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES,
}, default_flow_style=False))

print(f'\nDone. Dataset at: {out}')
print(f"  train: {len(splits['train'])} images")
print(f"  valid: {len(splits['valid'])} images")
print(f"  skipped (corrupt): {len(skipped)}")
if skipped:
    print('  first 5 skipped:', skipped[:5])

## Step 7 · Spot-check 20 random labels

If most boxes look right, proceed to Phase 2 (training).

If a class is consistently wrong (e.g. paint cans never detected, gloves confused with hands), **don't train yet** — tell me which class is bad and we'll either retune the prompt or fall back to manual seed-labeling for that class only.

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
from pathlib import Path

train_imgs = list(Path(OUTPUT_DIR, 'train', 'images').glob('*.jpg'))
random.seed(42)
sample = random.sample(train_imgs, min(20, len(train_imgs)))

PALETTE = [(255,80,80),(80,255,80),(80,80,255),(255,255,80),(255,80,255),
           (80,255,255),(255,160,0),(160,80,255),(0,200,160)]

fig, axes = plt.subplots(4, 5, figsize=(22, 14))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = Path(OUTPUT_DIR, 'train', 'labels', img_path.stem + '.txt')
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:5]]
            x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
            x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
            col = PALETTE[cls % len(PALETTE)]
            cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
            cv2.putText(img, CLASS_NAMES[cls], (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1, cv2.LINE_AA)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 8 · Class statistics

Bbox count per class. A class with `< 50` total labels means Grounding DINO struggled to find it — we'll need to retune that prompt before training.

In [ ]:
from collections import Counter
from pathlib import Path

counts = Counter()
img_with_labels = 0
img_total = 0
for split in ('train', 'valid'):
    img_total += len(list(Path(OUTPUT_DIR, split, 'images').glob('*.jpg')))
    for f in Path(OUTPUT_DIR, split, 'labels').glob('*.txt'):
        lines = [ln for ln in f.read_text().strip().splitlines() if ln.strip()]
        if lines:
            img_with_labels += 1
        for ln in lines:
            counts[int(ln.split()[0])] += 1

print(f"Images total       : {img_total}")
print(f"Images with labels : {img_with_labels}")
print(f"Empty labels       : {img_total - img_with_labels}")
print()
print(f"{'class':<12} {'count':>8}   status")
print('-' * 36)
for i, name in enumerate(CLASS_NAMES):
    n = counts.get(i, 0)
    if   n == 0:    flag = 'NONE FOUND'
    elif n < 50:    flag = 'low — retune prompt'
    elif n < 200:   flag = 'thin'
    else:           flag = 'ok'
    print(f'{name:<12} {n:>8}   {flag}')
print('-' * 36)
print(f"{'TOTAL':<12} {sum(counts.values()):>8}")

## Done — what to report back

Tell me:

1. **Total bboxes per class** (paste the table from Step 8).
2. **What looked right vs wrong** in the Step 7 grid — e.g. "paint cans never boxed", "gloves and hands sometimes overlap", "chairs perfect".

If everything looks reasonable → I'll write the Phase 2 training notebook.

If a class is broken → we retune that one prompt and re-run only the affected class (cheap, ~5 min).